In [14]:
# load modules and define path
import pandas as pd
from scipy.io import arff
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import shap

PATH = '../data/speeddating.arff'

## Data preparation


In [15]:
# view data
arff_file = arff.loadarff(PATH)
df = pd.DataFrame(arff_file[0])
df.head()

,has_null,wave,gender,age,age_o,d_age,d_d_age,race,race_o,samerace,...,d_expected_num_interested_in_me,d_expected_num_matches,like,guess_prob_liked,d_like,d_guess_prob_liked,met,decision,decision_o,match
0,b'0',1.0,b'female',21.0,27.0,6.0,b'[4-6]',b'Asian/Pacific Islander/Asian-American',b'European/Caucasian-American',b'0',...,b'[0-3]',b'[3-5]',7.0,6.0,b'[6-8]',b'[5-6]',0.0,b'1',b'0',b'0'
1,b'0',1.0,b'female',21.0,22.0,1.0,b'[0-1]',b'Asian/Pacific Islander/Asian-American',b'European/Caucasian-American',b'0',...,b'[0-3]',b'[3-5]',7.0,5.0,b'[6-8]',b'[5-6]',1.0,b'1',b'0',b'0'
2,b'1',1.0,b'female',21.0,22.0,1.0,b'[0-1]',b'Asian/Pacific Islander/Asian-American',b'Asian/Pacific Islander/Asian-American',b'1',...,b'[0-3]',b'[3-5]',7.0,NaN,b'[6-8]',b'[0-4]',1.0,b'1',b'1',b'1'
3,b'0',1.0,b'female',21.0,23.0,2.0,b'[2-3]',b'Asian/Pacific Islander/Asian-American',b'European/Caucasian-American',b'0',...,b'[0-3]',b'[3-5]',7.0,6.0,b'[6-8]',b'[5-6]',0.0,b'1',b'1',b'1'
4,b'0',1.0,b'female',21.0,24.0,3.0,b'[2-3]',b'Asian/Pacific Islander/Asian-American',b'Latino/Hispanic American',b'0',...,b'[0-3]',b'[3-5]',6.0,6.0,b'[6-8]',b'[5-6]',0.0,b'1',b'1',b'1'


In [16]:
print(list(df.columns))

['has_null', 'wave', 'gender', 'age', 'age_o', 'd_age', 'd_d_age', 'race', 'race_o', 'samerace', 'importance_same_race', 'importance_same_religion', 'd_importance_same_race', 'd_importance_same_religion', 'field', 'pref_o_attractive', 'pref_o_sincere', 'pref_o_intelligence', 'pref_o_funny', 'pref_o_ambitious', 'pref_o_shared_interests', 'd_pref_o_attractive', 'd_pref_o_sincere', 'd_pref_o_intelligence', 'd_pref_o_funny', 'd_pref_o_ambitious', 'd_pref_o_shared_interests', 'attractive_o', 'sinsere_o', 'intelligence_o', 'funny_o', 'ambitous_o', 'shared_interests_o', 'd_attractive_o', 'd_sinsere_o', 'd_intelligence_o', 'd_funny_o', 'd_ambitous_o', 'd_shared_interests_o', 'attractive_important', 'sincere_important', 'intellicence_important', 'funny_important', 'ambtition_important', 'shared_interests_important', 'd_attractive_important', 'd_sincere_important', 'd_intellicence_important', 'd_funny_important', 'd_ambtition_important', 'd_shared_interests_important', 'attractive', 'sincere', '

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8378 entries, 0 to 8377
Columns: 123 entries, has_null to match
dtypes: float64(59), object(64)
memory usage: 7.9+ MB


In [18]:
for col in df.columns:
    if df[col].dtype != "float64" :

        encode = OrdinalEncoder()
        encode.fit(df[[col]])

        df[col] = encode.fit_transform(df[[col]])
df.dropna(inplace = True)

In [19]:
# # drop 'match' and 'decision_o' column
# col_to_drop = ['match', 'decision_o', 'has_null', 'wave', 'expected_num_matches', 'd_expected_happy_with_sd_people', 'd_expected_num_matches', 'd_expected_num_interested_in_me', 'd_expected_num_matches', 'like', 'guess_prob_liked', 'd_like', 'd_guess_prob_liked', 'met']
# df.drop(col_to_drop, axis=1, inplace=True)

In [20]:
interest_cols = ['d_sports', 'd_tvsports', 'd_exercise', 'd_dining', 'd_museums', 'd_art', 'd_hiking', 'd_gaming',
    'd_clubbing', 'd_reading', 'd_tv', 'd_theater', 'd_movies', 'd_concerts', 'd_music', 'd_shopping',
    'd_yoga']
col_to_use = [
    'age', 'd_age', 'gender',
    'pref_o_sincere', 'pref_o_intelligence', 'pref_o_funny', 'pref_o_ambitious', 'pref_o_shared_interests',
    'sincere', 'intelligence', 'funny', 'ambition',
    'interests_correlate','decision_o',
] + interest_cols

df = df[col_to_use]
print(len(df))
df.head()

1048


,age,d_age,gender,pref_o_sincere,pref_o_intelligence,pref_o_funny,pref_o_ambitious,pref_o_shared_interests,sincere,intelligence,...,d_gaming,d_clubbing,d_reading,d_tv,d_theater,d_movies,d_concerts,d_music,d_shopping,d_yoga
0,21.0,6.0,0.0,20.0,20.0,20.0,0.0,5.0,8.0,8.0,...,0.0,0.0,1.0,2.0,0.0,2.0,2.0,2.0,1.0,0.0
1,21.0,1.0,0.0,0.0,0.0,40.0,0.0,0.0,8.0,8.0,...,0.0,0.0,1.0,2.0,0.0,2.0,2.0,2.0,1.0,0.0
3,21.0,2.0,0.0,5.0,15.0,40.0,5.0,5.0,8.0,8.0,...,0.0,0.0,1.0,2.0,0.0,2.0,2.0,2.0,1.0,0.0
4,21.0,3.0,0.0,10.0,20.0,10.0,10.0,20.0,8.0,8.0,...,0.0,0.0,1.0,2.0,0.0,2.0,2.0,2.0,1.0,0.0
5,21.0,4.0,0.0,0.0,30.0,10.0,0.0,10.0,8.0,8.0,...,0.0,0.0,1.0,2.0,0.0,2.0,2.0,2.0,1.0,0.0


In [21]:
# one hot encode 2x top both_like... and both_dislike... shared interestes
for col in interest_cols:
    like_name = col.replace('d_', 'both_like_')
    dislike_name = col.replace('d_', 'both_dislike_')
    df[like_name] = 0
    df[dislike_name] = 0

for row in df.itertuples():
    # make list of all values in interest_cols for this row
    interests = [getattr(row, col) for col in interest_cols]
    # get index of top 2 and bottom 2 values
    top_2_idx = sorted(range(len(interests)), key=lambda i: interests[i], reverse=True)[:2]
    bottom_2_idx = sorted(range(len(interests)), key=lambda i: interests[i])
    # get names of top_2_idx and bottom_2_idx
    top_2_cols = [interest_cols[i] for i in top_2_idx]
    bottom_2_cols = [interest_cols[i] for i in bottom_2_idx]
    # set like_name to 1 for top_2_cols and dislike_name to 1 for bottom_2_cols
    for col in top_2_cols:
        like_name = col.replace('d_', 'both_like_')
        df.at[row.Index, like_name] = 1
    for col in bottom_2_cols:
        dislike_name = col.replace('d_', 'both_dislike_')
        df.at[row.Index, dislike_name] = 1

# remove original interest columns
df.drop(interest_cols, axis=1, inplace=True)

In [22]:
df.columns

Index(['age', 'd_age', 'gender', 'pref_o_sincere', 'pref_o_intelligence',
       'pref_o_funny', 'pref_o_ambitious', 'pref_o_shared_interests',
       'sincere', 'intelligence', 'funny', 'ambition', 'interests_correlate',
       'decision_o', 'both_like_sports', 'both_dislike_sports',
       'both_like_tvsports', 'both_dislike_tvsports', 'both_like_exercise',
       'both_dislike_exercise', 'both_like_dining', 'both_dislike_dining',
       'both_like_museums', 'both_dislike_museums', 'both_like_art',
       'both_dislike_art', 'both_like_hiking', 'both_dislike_hiking',
       'both_like_gaming', 'both_dislike_gaming', 'both_like_clubbing',
       'both_dislike_clubbing', 'both_like_reading', 'both_dislike_reading',
       'both_like_tv', 'both_dislike_tv', 'both_like_theater',
       'both_dislike_theater', 'both_like_movies', 'both_dislike_movies',
       'both_like_concerts', 'both_dislike_concerts', 'both_like_music',
       'both_dislike_music', 'both_like_shopping', 'both_dislike_

In [23]:
[print(col) for col in df.columns];

age
d_age
gender
pref_o_sincere
pref_o_intelligence
pref_o_funny
pref_o_ambitious
pref_o_shared_interests
sincere
intelligence
funny
ambition
interests_correlate
decision_o
both_like_sports
both_dislike_sports
both_like_tvsports
both_dislike_tvsports
both_like_exercise
both_dislike_exercise
both_like_dining
both_dislike_dining
both_like_museums
both_dislike_museums
both_like_art
both_dislike_art
both_like_hiking
both_dislike_hiking
both_like_gaming
both_dislike_gaming
both_like_clubbing
both_dislike_clubbing
both_like_reading
both_dislike_reading
both_like_tv
both_dislike_tv
both_like_theater
both_dislike_theater
both_like_movies
both_dislike_movies
both_like_concerts
both_dislike_concerts
both_like_music
both_dislike_music
both_like_shopping
both_dislike_shopping
both_like_yoga
both_dislike_yoga


In [24]:
df.describe()

,age,d_age,gender,pref_o_sincere,pref_o_intelligence,pref_o_funny,pref_o_ambitious,pref_o_shared_interests,sincere,intelligence,...,both_like_movies,both_dislike_movies,both_like_concerts,both_dislike_concerts,both_like_music,both_dislike_music,both_like_shopping,both_dislike_shopping,both_like_yoga,both_dislike_yoga
count,1048.000000,1048.000000,1048.000000,1048.000000,1048.000000,1048.000000,1048.000000,1048.000000,1048.000000,1048.000000,...,1048.000000,1048.0,1048.000000,1048.0,1048.000000,1048.0,1048.000000,1048.0,1048.000000,1048.0
mean,25.005725,3.032443,0.493321,16.971021,22.255887,17.325029,9.725792,10.333626,8.158397,7.597328,...,0.096374,1.0,0.043893,1.0,0.040076,1.0,0.021947,1.0,0.007634,1.0
std,3.270365,2.427732,0.500194,7.450629,7.352106,6.666005,7.073420,6.763784,1.384077,1.771598,...,0.295244,0.0,0.204955,0.0,0.196232,0.0,0.146579,0.0,0.087078,0.0
min,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,2.000000,...,0.000000,1.0,0.000000,1.0,0.000000,1.0,0.000000,1.0,0.000000,1.0
25%,22.000000,1.000000,0.000000,10.000000,20.000000,10.832500,5.000000,5.000000,8.000000,7.000000,...,0.000000,1.0,0.000000,1.0,0.000000,1.0,0.000000,1.0,0.000000,1.0
50%,25.000000,2.000000,0.000000,18.000000,20.000000,18.180000,10.000000,10.000000,8.000000,8.000000,...,0.000000,1.0,0.000000,1.0,0.000000,1.0,0.000000,1.0,0.000000,1.0
75%,27.000000,4.000000,1.000000,20.000000,25.000000,20.000000,15.000000,15.000000,9.000000,9.000000,...,0.000000,1.0,0.000000,1.0,0.000000,1.0,0.000000,1.0,0.000000,1.0
max,35.000000,14.000000,1.000000,40.000000,50.000000,40.000000,53.000000,30.000000,10.000000,10.000000,...,1.000000,1.0,1.000000,1.0,1.000000,1.0,1.000000,1.0,1.000000,1.0


In [25]:
X = df.drop('decision_o', axis=1)
y = df['decision_o']

# scale the data, because the dataset contains features with different scales, which can negatively impact the performance of some machine learning algorithms.
scaler = StandardScaler()

# create seperate train sets, because the dataset only contains hereteosexual matches, which means that the personlized model will only be trained on men or women.
X_women = X[X['gender'] == 0]
X_women_scaled = scaler.fit_transform(X_women.drop('gender', axis=1))
y_women = y[X['gender'] == 0]
X_men = X[X['gender'] == 1]
X_men_scaled = scaler.fit_transform(X_men.drop('gender', axis=1))
y_men = y[X['gender'] == 1]

# drop gender
X_women.drop('gender', axis=1, inplace=True)
X_men.drop('gender', axis=1, inplace=True)  

In [26]:
X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(X_women, y_women, test_size=0.2, random_state=42)
X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(X_men, y_men, test_size=0.2, random_state=42)
# create scaled versions (after train-test split to avoid data leakage)
X_w_train_scaled = scaler.fit_transform(X_w_train)
X_w_test_scaled = scaler.transform(X_w_test)
X_m_train_scaled = scaler.fit_transform(X_m_train)
X_m_test_scaled = scaler.transform(X_m_test)

print(len(X_w_train))

424


In [27]:
# create mini dataset to simulate the data size of the experiment we are going to do
X_w_train_mini = X_w_train[:60]
X_w_train_scaled_mini = X_w_train_scaled[:60]
y_w_train_mini = y_w_train[:60]

X_m_train_mini = X_m_train[:60]
X_m_train_scaled_mini = X_m_train_scaled[:60]
y_m_train_mini = y_m_train[:60]

## Training

- I don't expect shap explanation to be very insightful, because the data is from different participants (as there is no way to distinguish them). What we might see is that common dating preferences appear in the explanations.
- TODO: the number of features still need to be decreased, by manual selection.

In [28]:
X_train_sel = X_w_train_scaled_mini
y_train_sel = y_w_train_mini
X_test_sel = X_w_test_scaled
y_test_sel = y_w_test

In [29]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=500)
model.fit(X_train_sel, y_train_sel)
accuracy = model.score(X_test_sel, y_test_sel)
print(f'The accuracy is: {accuracy}')


The accuracy is: 0.5794392523364486


In [30]:
from sklearn.inspection import PartialDependenceDisplay
from sklearn.preprocessing import MinMaxScaler
import numpy as np

features = ['sincere',
            'intelligence',
            'funny',
            'ambition',
            'interests_correlate'      # tells us level of shared interest
            ]
features_idx = [X_w_train.columns.get_loc(feature) for feature in features]   
coeficients = []

for idx in features_idx:
    coeficients.append(float(model.coef_[0][idx]))
    print(f'Coefficient for feature {X_w_train.columns[idx]}: {model.coef_[0][idx]:.2f}')

# normalize coeficients between -1 and 1
scaler = MinMaxScaler(feature_range=(-1, 1))
coeficients = scaler.fit_transform(np.array(coeficients).reshape(-1, 1)).flatten()
print(f'Normalized coefficients: {coeficients}')

Coefficient for feature sincere: -0.77
Coefficient for feature intelligence: -0.15
Coefficient for feature funny: 0.17
Coefficient for feature ambition: -0.65
Coefficient for feature interests_correlate: -0.18
Normalized coefficients: [-1.          0.32321215  1.         -0.74514393  0.25993432]


In [31]:
# not used anymore

PartialDependenceDisplay.from_estimator(model, X_test_sel, features_idx, kind='average')

ImportError: PartialDependenceDisplay.from_estimator requires matplotlib. You can install matplotlib with `pip install matplotlib`

In [ ]:
# not used anymore

explainer = shap.Explainer(model, X_train_sel)
shap_values = explainer(X_test_sel)
# shap.waterfall_plot(shap_values[0])
shap.summary_plot(shap_values, X_test_sel)
print(f'The accuracy of the explainer model is: {accuracy}')

## Other models (not used anymore)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train_sel, y_train_sel)
accuracy_dt = model_dt.score(X_test_sel, y_test_sel)
print(f'The accuracy of decision tree is: {accuracy_dt:.3f}')

# print the feature importance of the decision tree model
feature_importance = model_dt.feature_importances_
if hasattr(X_train_sel, "columns"):
    feature_names = X_train_sel.columns
else:
    feature_names = X_w_train.columns  # feature order matches the NumPy array

for col, importance in zip(feature_names, feature_importance):
    if importance > 0:
        print(f"{col}: {importance:.2f}")

explainer = shap.Explainer(model_dt, X_train_sel)
shap_values = explainer(X_test_sel)
shap.summary_plot(shap_values, X_test_sel)
print(f'The accuracy of the explainer model is: {accuracy_dt:.3f}')

In [ ]:
# random forest
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train_sel, y_train_sel)
accuracy_rf = model_rf.score(X_test_sel, y_test_sel)
print(f'The accuracy of random forest is: {accuracy_rf:.3f}')

explainer = shap.Explainer(model_rf, X_train_sel,)
shap_values = explainer(X_test_sel, check_additivity=False)
shap.summary_plot(shap_values, X_test_sel)
print(f'The accuracy of the explainer model is: {accuracy_rf:.3f}')

In [ ]:
from xgboost import XGBClassifier

model_xgb = XGBClassifier(random_state=42)
model_xgb.fit(X_train_sel, y_train_sel)
accuracy_xgb = model_xgb.score(X_test_sel, y_test_sel)
print(f'The accuracy of xgboost is: {accuracy_xgb:.3f}')

explainer = shap.Explainer(model_xgb, X_train_sel)
shap_values = explainer(X_test_sel)
shap.summary_plot(shap_values, X_test_sel)
print(f'The accuracy of the explainer model is: {accuracy_xgb:.3f}')

### Notes
*using `X_w_train_mini` / `X_w_train_scaled_mini` and `y_w_train_mini` of size 60* using standard scaling

**Logistic model**
- The SHAP explainer model and the logistic model have the exact same accuracy, which means that the SHAP model probably works the same as the logistic model, so the explanations must be accurate.
- This model performed the best out of all, given that we use `X_w_train_scaled_mini`

**Decision tree /  Random Forest**
- The decision tree / Random Forest model is too simple for the problem at hand. It does not allow for enough complexity to get a decent accuracy, given a small dataset. We should not use this model.
- The SHAP explanation is terrible because the model is too simple.

**XGBoost**
- Better than logistic model

**Other**
- For non-black-box models, model native explanation might be better. Like regularized coefficients from the logistic model.
- All models achieve higher accuracy given more data, so this might be an issue in our project. However, is all data is from the same person (with the same preferences) than we probably will get a higher accuracy with less data.
- Use waterfall (instead of summary) plots to visualize the SHAP explanation for a single prediction.
